# Решение задачи кредитного скоринга 
Решение задачи кредитного скоринга будет представлено в нескольких ноутбуках. В данном ноутбуке будет реалиован SHAP - интерпретация признаков модели.

## Данные
Датасет соревнования устроен таким образом, что кредиты для тренировочной выборки взяты за период в М месяцев, а кредиты для тестовой выборки взяты за последующие K месяцев.

Каждая запись кредитной истории содержит самую разнообразную информацию о прошлом кредите клиента, например, сумму, отношение клиента к кредиту, дату открытия и закрытия, информацию о просрочках по платежам и др. Все публикуемые данные тщательно анонимизированы.

Целевая переменная – бинарная величина, принимающая значения 0 и 1, где 1 соответствует дефолту клиента по кредиту.

## Проверка решения
Итоговая метрика решения - ROC AUC. Решения принимаются в виде csv-файла с двумя колонками: “id” – уникальный идентификатор и “score” – предсказание дефолта клиента по кредиту.

## Импорт библиотек

In [ ]:
# Импорт библиотек
import json
import pickle
import sys

sys.path.append('../src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from xgboost import XGBClassifier



## Чтение датасета и подготовка данных
Прочитаем сагрегированный датасет, а также загрузим найденные гиперпараметры для модели и отобранные признаки.

In [3]:
train = pd.read_parquet('../artifacts/aggregated_df.parquet')
train.head(2)

,flag,n_loans,pre_since_opened_mean,pre_since_opened_max,pre_since_opened_sum,pre_since_opened_std,pre_since_confirmed_mean,pre_since_confirmed_max,pre_since_confirmed_sum,pre_since_confirmed_std,...,enc_paym_20_mean,enc_paym_20_max,enc_paym_21_mean,enc_paym_21_max,enc_paym_22_mean,enc_paym_22_max,enc_paym_23_mean,enc_paym_23_max,enc_paym_24_mean,enc_paym_24_max
id,,,,,,,,,,,,,,,,,,,,,
0,0,10,8.100000,18,81,7.030884,7.600000,12,76,3.864367,...,3.400000,4,2.400000,3,2.400000,3,2.400000,3,4.000000,4
1,0,14,11.428571,18,160,4.182643,7.642857,14,107,3.855209,...,3.142857,4,2.142857,3,2.428571,3,2.357143,3,3.357143,4


In [5]:
# Фиксируем признаки, которые будут использоваться в нашей модели
FEATURES = list(pd.read_csv('../artifacts/features.csv', index_col=0)['feature'])
FEATURES

['enc_loans_account_cur_nunique',
 'enc_loans_account_holder_type_nunique',
 'enc_loans_credit_status_mode_value',
 'enc_loans_credit_type_mode_value',
 'enc_loans_credit_type_nunique',
 'enc_paym_0_max',
 'enc_paym_0_mean',
 'enc_paym_13_mean',
 'enc_paym_15_mean',
 'enc_paym_16_mean',
 'enc_paym_17_mean',
 'enc_paym_18_mean',
 'enc_paym_19_mean',
 'enc_paym_1_max',
 'enc_paym_1_mean',
 'enc_paym_20_mean',
 'enc_paym_21_mean',
 'enc_paym_22_mean',
 'enc_paym_23_mean',
 'enc_paym_2_mean',
 'enc_paym_3_mean',
 'enc_paym_4_mean',
 'enc_paym_5_mean',
 'enc_paym_9_mean',
 'is_zero_loans3060_mean',
 'is_zero_loans530_mean',
 'is_zero_loans5_mean',
 'is_zero_loans6090_mean',
 'is_zero_loans90_mean',
 'is_zero_over2limit_mean',
 'is_zero_util_mean',
 'pre_loans5_std',
 'pre_loans_credit_cost_rate_max',
 'pre_loans_credit_cost_rate_mean',
 'pre_loans_credit_cost_rate_std',
 'pre_loans_credit_limit_max',
 'pre_loans_max_overdue_sum_mean',
 'pre_loans_max_overdue_sum_std',
 'pre_loans_next_pay_s

In [6]:
# Фиксируем найденные гиперпараметры для модели
with open('../artifacts/best_params.json', 'r', encoding='utf-8') as f:
    best_params = json.load(f)
    
best_params

{'colsample_bytree': 0.6529591345528285,
 'learning_rate': 0.07233726319671857,
 'max_depth': 5,
 'min_child_weight': 2,
 'n_estimators': 559,
 'reg_lambda': 0.9536440720641501,
 'subsample': 0.9703638562674948}

## SHAP - интепретация признаков
Используем библиотеку SHAP для интерпетации признаков нашей модели. А также посмотрим на конкретных примерах, почему было принято то или иное решение по дефолту.